In [2]:
from langchain_openai import ChatOpenAI

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
    api_key=OPENAI_API_KEY
)

In [4]:
response=model.invoke('what are skills of rathan sumbet')
response.content

"It appears you may be referring to someone named **Rathan Sumbet**. However, as of my latest knowledge update (June 2024), there is no widely recognized public figure, professional, artist, or athlete known by the name **Rathan Sumbet**. It's possible there may be a typo in the name or the individual could be a private person or an emerging figure who hasn't gained major coverage yet.\n\nIf you could provide more context—such as their profession, industry, or correct spelling—I would be better able to help you list their skills or provide relevant information.\n\n**Please clarify or provide more details, and I’ll do my best to assist!**"

## Buliding RAG resume

In [5]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


c:\Users\Hello\Innomatics\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
file_path = r"C:\Users\Hello\Downloads\20605119-Resumes"

loader = DirectoryLoader(
    file_path,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

docs = loader.load()

## chucking

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
splitter=RecursiveCharacterTextSplitter(separators=['\n\n','\n',' ',''],
                                        chunk_size=600,
                                        chunk_overlap=100)
chunks=splitter.split_documents(docs)
print(f'the no of chunks are {len(chunks)}')

the no of chunks are 444


## embedding

In [9]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=OPENAI_API_KEY,
    dimensions=1000
)

## vectore store

In [10]:
from langchain_chroma import Chroma

In [11]:
vectore_store=Chroma(embedding_function=embedding,persist_directory='ResumeRAG_1000')

In [12]:
vectore_store.add_documents(chunks)

['42487c18-ee22-4411-8611-2dba57f0af72',
 '5aea2632-ac42-48aa-884b-169e81b638ed',
 '9a8f20e0-1f3d-4dcb-bc7a-d43371bb5bc7',
 '713bc30a-4fff-48a4-b26f-f8cf35a86d13',
 '9b6efd67-767d-423a-9ea9-4feb7579d445',
 'cc5e3644-5e36-4190-b6f9-c9a42f30215d',
 'c4a8176e-a8fc-4a60-9711-962b40abec51',
 '81086861-97aa-40e2-9bdc-60b3c3bde0af',
 'b459ba29-b8ec-4c24-b0e3-4de28208348e',
 'c4781548-f288-4131-88f7-dc65865886f3',
 '8a56574f-da08-430d-a61d-2fb7cf52b4e1',
 '0fdbb1c9-2389-4188-be80-05f5a15a4e06',
 'dd06dd44-a6a8-4fc3-aaa6-3b34dd4247bb',
 'dafb3b7b-107f-45a9-a02f-269387d0f165',
 '0b257a25-2299-4c5e-b3b4-30c85fdf4721',
 '31f9628a-baed-4f78-bb52-2c2b0950d904',
 'dc93c4e6-6f5e-4139-850b-1fabf2fcc311',
 'b4fd3c84-9399-4c2e-87ce-5ce25ff38775',
 '7ecfa2f9-0fb8-4c49-9b03-c0e9e50d693d',
 '406e1b14-3b59-4fe9-966b-53759d3211dc',
 'fed7ab96-a8f9-4ab4-9d35-64cfd96337df',
 'bf68a67b-9252-4730-8e97-d73e46157389',
 '566ba160-ef1c-4b6b-a787-62878b680193',
 '7003a15a-7de5-4407-b739-5b6656e5afe5',
 'd6075288-bd29-

## Retrievers

In [13]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [14]:
retriever_similarity = vectore_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 30}
)

In [15]:
query='what is the phone number of Tejaswi?'
retrieved_doc=retriever_similarity.invoke(query)

In [16]:
context_text= "\n\n".join(
    f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}"
    for doc in retrieved_doc
)

In [17]:
print(context_text)

[Source: C:\Users\Hello\Downloads\20605119-Resumes\Resumes\Tejaswi resume.pdf]
TEJASWI SIDDAGONI
Hyderabad, Telangana, India  |  +91-9000000567  |  tejaswi@gmail.com
linkedin.com.in  |  github.com
PROFESSIONAL SUMMARY
B.Tech (AI/ML) graduate with hands-on experience building Power BI dashboards, SQL-based business analysis, Python  
applications, and AI/LLM solutions. Experienced in turning raw data into reports and building practical tools using Python, 
SQL, Flask, Streamlit, LangChain, and LLM APIs. Targeting entry-level technical roles in AI, business applications,  
automation, and analytics.
TECHNICAL SKILLS
Programming: Python, SQL

[Source: C:\Users\Hello\Downloads\20605119-Resumes\Resumes\Tejaswi resume.pdf]
TEJASWI SIDDAGONI
Hyderabad, Telangana, India  |  +91-9000000567  |  tejaswi@gmail.com
linkedin.com.in  |  github.com
PROFESSIONAL SUMMARY
B.Tech (AI/ML) graduate with hands-on experience building Power BI dashboards, SQL-based business analysis, Python  
applications, and

### augumentation

In [18]:
SYSTEM_PROMPT = """
You are ResumeIQ — an intelligent, reliable, and context-grounded AI Resume
Retrieval Assistant.

Your goal is to transform unstructured resume information into useful,
accurate, and recruiter-friendly insights.

You will receive:
1. A user query or recruitment requirement.
2. Resume information retrieved from a vector database.

========================
CORE PRINCIPLES
========================

1. GROUNDED INTELLIGENCE
Use ONLY the information available in the retrieved resume context.
Never invent skills, experience, education, certifications, projects,
companies, or achievements.

2. RELEVANCE FIRST
Identify the resume information that is directly relevant to the user's
query. Ignore unrelated information.

3. MULTI-CANDIDATE RETRIEVAL
If multiple candidates match the query, identify and present all relevant
candidates instead of returning only one candidate.

4. EVIDENCE-BASED ANSWERS
Every candidate insight must be supported by the retrieved resume content.
Do not make assumptions based on job titles or common industry expectations.

5. TRANSPARENT UNCERTAINTY
If the retrieved context does not contain enough information to answer the
question confidently, clearly state:

"I couldn't find sufficient information in the retrieved resumes."

Do not guess.

========================
RESPONSE INTELLIGENCE
========================

For candidate searches:
- Mention the candidate name or resume source when available.
- Highlight the skills, education, experience, projects, and technologies
  relevant to the query.
- Keep the response concise and recruiter-friendly.
- Use bullet points or a structured format when multiple candidates are found.

For skill-based queries:
- Identify candidates whose retrieved resume content explicitly contains
  the requested skill or closely relevant information.
- Do not treat an unstated skill as present.

For comparison queries:
- Present candidates in a clear and structured format.
- Compare only information supported by the retrieved resumes.

========================
RECRUITER-FRIENDLY OUTPUT
========================

Prioritize:
✓ Accuracy
✓ Relevance
✓ Evidence
✓ Candidate identification
✓ Concise explanations

Avoid:
✗ Hallucinations
✗ Unsupported assumptions
✗ Irrelevant resume information
✗ Long copied passages

========================
RETRIEVED RESUME CONTEXT
========================

{context}

========================
USER QUERY
========================

{query}

========================
FINAL ANSWER
========================
"""

In [19]:
system_prompt = SYSTEM_PROMPT.format(
    context=context_text,
    query=query
)

response = model.invoke(system_prompt)

print(response.content)

The phone number of Tejaswi Siddagoni is +91-9000000567.


In [20]:
response=model.invoke(SYSTEM_PROMPT)
print(response.content)

I couldn't find sufficient information in the retrieved resumes.


In [21]:
def RAG_SYSTEM(query):
    
    retrieved_doc = retriever_similarity.invoke(query)

    context_text = '\n\n'.join(
        f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in retrieved_doc
    )

    SYSTEM_PROMPT = f"""
You are ResumeIQ, an intelligent AI Resume Retrieval Assistant.

Your task is to answer user questions using ONLY the resume information
provided in the retrieved context below.

======================
GROUNDING RULES
======================

1. Treat the retrieved resume context as the single source of truth.

2. Do not invent, assume, or infer any candidate information that is not
   explicitly available in the retrieved resumes.

3. Carefully analyze ALL retrieved resume excerpts before answering.

4. If multiple candidates match the user's query, return all relevant
   candidates rather than selecting only one.

5. Mention the candidate name or resume filename whenever it is available.

6. Distinguish between information that is explicitly present and
   information that cannot be determined from the retrieved resumes.

7. If the retrieved context does not contain enough information to answer
   the question, respond exactly:

   "I couldn't find sufficient information in the retrieved resumes."

======================
RESPONSE STYLE
======================

- Be concise, factual, and recruiter-friendly.
- Use bullet points when presenting multiple candidates.
- Highlight only information relevant to the user's question.
- Do not copy large portions of resumes.
- Do not make assumptions based on job titles or common industry knowledge.
- If the retrieved context is incomplete, clearly mention the limitation.

======================
RETRIEVED RESUME CONTEXT
======================

{context_text}

======================
USER QUERY
======================

{query}

======================
GROUNDED ANSWER
======================
"""

    response = model.invoke(SYSTEM_PROMPT)

    print(response.content)
    

In [22]:
RAG_SYSTEM(query='what are the skills of Tejaswi')

- Candidate: Tejaswi Siddagoni
- Skills explicitly mentioned:
  - Python
  - SQL
  - Power BI dashboards
  - SQL-based business analysis
  - Python applications
  - AI/LLM solutions
  - Flask
  - Streamlit
  - LangChain
  - LLM APIs

These are the technical skills listed in Tejaswi Siddagoni's resume.


In [23]:
user_input = """
Develop and implement AI/ML solutions and intelligent automation workflows.

• Build and integrate AI Agents and LLM-based applications.
• Work with WhatsApp CRM and WhatsApp automation.
• Develop Instagram automation and social media workflow solutions.
• Work on Threads scraping and automation.
• Develop and maintain website downtime monitoring tools.
• Create and configure Meta Apps and work with Meta APIs.
• Integrate third-party APIs and automation platforms into business workflows.
• Work with web scraping, data extraction, and automation systems.
• Explore and implement AI-powered image generation solutions.
• Develop automation workflows for repetitive business processes.
• Test, troubleshoot, optimize, and deploy AI/ML and automation solutions.
• Work with databases, APIs, webhooks, and external integrations.
• Research emerging AI tools and technologies and identify ways to implement
  them in real-world projects.
"""

In [24]:
RAG_SYSTEM(
    query=f"""
Identify the students who are familiar with the given job description.

Job Description:
{user_input}
"""
)

Based on the retrieved resumes, the following candidates demonstrate familiarity with multiple aspects of the provided job description:

---

**1. Mohammed Mudasir - Mudasir Ahmed**
- Experience in developing and deploying AI/ML solutions, including automation workflows.
- Has built AI agents, intelligent chatbots, and autonomous agents.
- Hands-on experience with Python, LLMs, Generative AI, and implementing RAG-based applications.
- Experience with predictive models (classification, regression, anomaly detection) and Generative AI applications.
- Specialist in automation to reduce manual effort and resolution times.
- Has worked on integrating AI into enterprise workflows and deploying solutions that handle incident triage and FAQs.
- Experience includes deploying AI-powered assistants, which aligns with automation and agent-based system requirements.
- No explicit mention of social media automation, WhatsApp CRM, Instagram automation, Meta APIs, Threads automation, or image generati

In [25]:
import json
import re

def evaluate_candidate(job_description, candidate_name, candidate_resume):

    prompt = f"""
You are an expert AI recruitment evaluator.

Evaluate the candidate ONLY against the provided Job Description.

Do not assume or invent any information that is not explicitly present
in the candidate's resume.

JOB DESCRIPTION:
{job_description}

CANDIDATE NAME:
{candidate_name}

CANDIDATE RESUME:
{candidate_resume}

Evaluate the candidate based on:
- Required skills
- Technical skills
- Tools and technologies
- Relevant experience
- Projects
- Education, when relevant

Return ONLY valid JSON in exactly this format:

{{
    "candidate": "{candidate_name}",
    "match_score": 0,
    "matching_skills": [],
    "missing_skills": [],
    "experience_fit": "",
    "reason": ""
}}

Scoring:
90-100 = Excellent match
80-89 = Strong match
70-79 = Good match
60-69 = Partial match
Below 60 = Weak match

The match_score must be between 0 and 100.
"""

    response = model.invoke(prompt)

    result_text = response.content

    # Remove markdown code fences if the model adds them
    result_text = re.sub(
        r"```json|```",
        "",
        result_text
    ).strip()

    return json.loads(result_text)

In [26]:
retrieved_docs = retriever_similarity.invoke(
    f"""
    Identify candidates relevant to this Job Description:

    {user_input}
    """
)

In [27]:
candidate_resumes = {}

for doc in retrieved_docs:

    source = doc.metadata.get("source")

    if source not in candidate_resumes:
        candidate_resumes[source] = []

    candidate_resumes[source].append(
        doc.page_content
    )

In [28]:
print("Unique candidates:", len(candidate_resumes))

Unique candidates: 2


In [29]:
candidate_results = []

for source, chunks in candidate_resumes.items():

    candidate_resume = "\n\n".join(chunks)

    candidate_name = source.split("\\")[-1]

    result = evaluate_candidate(
        job_description=user_input,
        candidate_name=candidate_name,
        candidate_resume=candidate_resume
    )

    result["source"] = source

    candidate_results.append(result)

In [30]:
ranked_candidates = sorted(
    candidate_results,
    key=lambda x: x["match_score"],
    reverse=True
)

In [31]:
for rank, candidate in enumerate(ranked_candidates, 1):

    print(f"\nRank {rank}")
    print("Candidate:", candidate["candidate"])
    print("Match Score:", candidate["match_score"], "%")
    print(
        "Matching Skills:",
        ", ".join(candidate["matching_skills"])
    )
    print(
        "Missing Skills:",
        ", ".join(candidate["missing_skills"])
    )
    print("Experience Fit:", candidate["experience_fit"])
    print("Reason:", candidate["reason"])
    print("-" * 70)


Rank 1
Candidate: Bangaru_SrikaraResume.pdf
Match Score: 68 %
Matching Skills: Python, Machine Learning, LLMs, LangChain, OpenAI API, Workflow Automation, Rapid Prototyping, Chatbot Development, Prompt Engineering, Process Discovery, AI Dashboards, Data Analysis, Scikit-learn, Diffusion Models
Missing Skills: WhatsApp CRM and WhatsApp automation, Instagram automation, Threads scraping and automation, Website downtime monitoring tools, Meta Apps and Meta APIs, Integration with third-party APIs and automation platforms, Web scraping, Data extraction, Website monitoring, AI-powered image generation solutions (implementation), Databases, APIs, webhooks and external integration (explicit experience), Testing, troubleshooting, optimizing and deploying AI/ML and automation solutions, Social media workflow solutions
Experience Fit: The candidate demonstrates strong experience in generative AI, LLMs, Python, workflow automation, and related AI/ML tools and techniques. However, there is no expl

In [32]:
shortlisted_candidates = ranked_candidates[:4]

In [33]:
print("\n========== SHORTLISTED CANDIDATES ==========")

for rank, candidate in enumerate(shortlisted_candidates, 1):

    print(
        f"{rank}. "
        f"{candidate['candidate']} - "
        f"{candidate['match_score']}%"
    )


========== SHORTLISTED CANDIDATES ==========
1. Bangaru_SrikaraResume.pdf - 68%
2. Mohammed_Mudasir - Mudasir Ahmed.pdf - 67%


## RAG eavalution

In [36]:
%pip install ragas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import ragas

In [41]:
from ragas import evaluate

In [36]:
from ragas.metrics import (
    answer_correctness,
    answer_relevancy,
    context_precision,
    context_recall,
)

In [37]:
questions = [
    "What LangChain and OpenAI based projects has Akshitha Manne built?",
    "What is Akshitha Manne's educational background?",
    "What accuracy did Akshitha Manne's Fake News Detection project achieve?",
    "What NLP techniques and libraries does Akshitha Manne have experience with?",
    "What accuracy did Bangaru Srikara Datta's gesture-controlled virtual calculator achieve?",
    "What R² score did Srikara's delivery time prediction model achieve?",
    "What is Bangaru Srikara Datta's educational background?",
    "What computer vision tools has Srikara used and for what purpose?",
    "What is Elluri Bhargavi's educational background?",
    "How many rental listings did Bhargavi analyze in her rental price transparency project?",
    "What is Elluri Bhargavi's current role?",
    "What did Bhargavi's Finance Performance Dashboard project involve?",
    "What accuracy did Vijaya Lakshmi Bollapalli's Uber ride booking prediction achieve?",
    "What is Vijaya Lakshmi Bollapalli's educational background?",
    "What Gen AI skills does Vijaya Lakshmi Bollapalli list?",
    "What accuracy did Vijaya Lakshmi's fake news detection model achieve?",
    "What accuracy did Chayan Chuhan's sentiment analysis project achieve?",
    "What R² did Chayan Chuhan's Swiggy delivery time prediction achieve?",
    "What frameworks does Chayan Chuhan use for building APIs?",
    "What is Chayan Chuhan's educational background?",
    "What is this candidate's educational background?",
    "What SQL project did this candidate build?",
    "What publications has this candidate authored?",
    "What did this candidate's credit card fraud detection project involve?",
    "What tech stack did Praneeth Goud use for the Education Tutor Chatbot?",
    "What is Edulakanti Praneeth Goud's educational background?",
    "What did Praneeth Goud's IMDB movie review project involve?",
    "What Gen AI skills does Praneeth Goud list?",
    "What is Hema Goud's educational background?",
    "What accuracy did Hema Goud's mental health text classification project achieve?",
    "What languages does Hema Goud speak?",
    "What accuracy did Hema Goud's fake news detection project achieve?",
    "What dataset did Shahina use for her EV market analysis project?",
    "What accuracy did Shahina's Flipkart washing machine data scraping project achieve?",
    "What is Kalagotla Shahina's educational background?",
    "What models did Shahina use in her Swiggy delivery time prediction project?",
    "What LLM did Chintha Kalyan use for the AI Gym Trainer Assistant?",
    "What is Chintha Kalyan's educational background?",
    "What did Chintha Kalyan's mobile price prediction project involve?",
    "What Gen AI and NLP skills does Chintha Kalyan have?",
    "What R² and RMSE did Kandela Vamshi's XGBoost delivery prediction model achieve?",
    "What technique did Kandela Vamshi use for the movie recommendation system?",
    "What accuracy did Kandela Vamshi's virtual keyboard project achieve?",
    "What is Kandela Vamshi's educational background?",
    "What did Lekshana Priya build during her internship at Auzmor Technology?",
    "What is Lekshana Priya R's educational background?",
    "What multi-agent system did Lekshana Priya design in RupeeRadar?",
    "What R² score did Manduri Kushal's delivery time prediction achieve?",
    "What did Manduri Kushal build for online proctoring?",
    "What is Manduri Kushal Virendra Chowdary's educational background?"
]

In [44]:
ground_truths = [
    "Akshitha built a Conversational AI Chatbot using LangChain, OpenAI GPT, and ConversationBufferMemory with prompt engineering and LLMChain for multi-turn interactions, and a Zomato AI Restaurant Assistant using LangChain and OpenAI GPT-4o Mini.",
    "Akshitha completed a Data Science program at Innomatics Research Science (07/2024) and holds a Bachelor of Technology in Electronics and Instrumentation from Vignan Institute of Technology and Science (09/2021–07/2025).",
    "Akshitha's Fake News Detection system, built using RNN and NLP on 40K+ news articles, achieved 85% classification accuracy.",
    "Akshitha has experience with NLTK, spaCy, Bag-of-Words, TF-IDF, GloVe, FastText, RNNs, LSTMs, Auto Encoders, and BERT.",
    "Srikara's Gesture-Controlled Virtual Calculator achieved 99.91% gesture-to-text accuracy.",
    "Srikara's Decision Tree Regressor for delivery time prediction achieved an R² of 0.77 on the test set with low MAE.",
    "Srikara holds a Bachelor of Technology in Petroleum Engineering from Vignan's University for Science and Technology, Guntur (2020–2024), with a CGPA of 8.0/10.0.",
    "Srikara has used OpenCV and MediaPipe for real-time gesture recognition and multimodal AI applications.",
    "Bhargavi holds an MBA in Finance from Nobel Post Graduate College (2023–2025, CGPA 9.15) and a B.Com from Jagruthi Women College (2020–2023, CGPA 8.90).",
    "Bhargavi scraped and analyzed 1,200+ rental listings in Hyderabad for her Rental Price Transparency Analysis project.",
    "Bhargavi currently works as a Data Science Trainee at Innomatics Research Labs (2025–Present).",
    "Bhargavi built a Power BI dashboard analyzing 5,000+ financial records, tracking revenue, expenses, and profitability KPIs across 3 financial statements, with 10+ DAX-based visualizations for growth rates and profit margins.",
    "Vijaya Lakshmi's ANN model for Uber Ride Booking Status Prediction, using SMOTE and Optuna tuning, achieved 91% accuracy on 150,000+ ride records.",
    "Vijaya Lakshmi holds an M.Sc in Data Science from Bankatlal Badruka College for Information and Technology (2023–2025, CGPA 7.20) and a B.Sc in Computer Science from Andhra Mahila Sabha Arts and Science College for Women (2020–2023, CGPA 9.25).",
    "Vijaya Lakshmi's Gen AI skills include LLMs, OpenAI, Gemini, LangChain, Prompt Engineering, and RAG.",
    "Vijaya Lakshmi's Bi-LSTM model with Word2Vec embeddings achieved 95%+ classification accuracy on unseen data for fake news detection across 40,000+ news articles.",
    "Chayan's Bidirectional LSTM sentiment analysis model on 205K+ Flipkart reviews achieved ~92% validation accuracy.",
    "Chayan's tuned XGBoost regressor for Swiggy delivery time prediction achieved an R² of 0.82 on held-out test data.",
    "Chayan uses FastAPI, Streamlit, Pydantic, and Uvicorn for building APIs.",
    "Chayan holds a B.Tech in Information Technology from Aditya Silver Oak Institute of Technology, Ahmedabad (2021–2025), with a CGPA of 8.64/10.",
    "The candidate holds a Bachelor of Technology in Computer Science and Engineering with an AI & ML specialization from KNRR Engineering College (12/2021–07/2025).",
    "The candidate built a Hospital Management System: a relational MySQL database covering patients, doctors, appointments, treatments, and billing, using joins, aggregations, grouping, and filtering to analyze appointments, revenue, payment status, and doctor performance.",
    "The candidate authored two publications: 'Understanding Discrete Probability Distributions: From Coin Tosses to Customer Calls' (16/11/2025) and 'From Real World to Code: Understanding OOPs' (10/09/2025).",
    "The candidate developed an ANN-based classification model to detect fraudulent credit card transactions from imbalanced data, applying feature scaling and class-imbalance handling, and evaluated results using precision, recall, F1-score, and a confusion matrix.",
    "Praneeth built the Education Tutor Chatbot using Python, LangChain, OpenAI, Streamlit, Prompt Engineering, Generative AI, Git, and GitHub.",
    "Praneeth is pursuing a Data Science Program at Innomatics Research Labs (09/2025–Present) and holds a B.Tech in CSE (AI & ML) from Gitam University, Hyderabad (2021–2025).",
    "Praneeth built an LSTM-based sentiment analysis system classifying IMDB reviews as Positive or Negative, trained with TensorFlow/Keras, and deployed via a Streamlit app on Hugging Face with real-time predictions and confidence scores.",
    "Praneeth's Gen AI skills include LLMs, Prompt Engineering, LangChain, OpenAI API, Hugging Face, RAG, and Fine Tuning.",
    "Hema completed a Data Science program at Innomatics Research Science (09/2025) and holds a BSc in Forensic Science from Kalinga University (06/2022–05/2025).",
    "Hema's Bidirectional LSTM mental health text classification system achieved 80% accuracy across 7 categories on 50K+ text records.",
    "Hema speaks English, Hindi, and Telugu.",
    "Hema's RNN-based Fake News Detection system achieved 85% classification accuracy on 40K+ news articles.",
    "Shahina used global electric vehicle (EV) datasets from the IEA (International Energy Agency), 2024, for her EV market analysis project.",
    "Shahina's Flipkart washing machine scraping and analysis pipeline achieved 95–98% data accuracy using BeautifulSoup, Requests, Matplotlib, and Seaborn.",
    "Shahina studied Data Science and Gen AI at Innomatics Research Science (06/2025–03/2026) and holds a degree in Electronics and Communication Engineering from Vignan Lara Institute of Technology and Science (12/2021–04/2025).",
    "Shahina built Linear Regression, Decision Tree, and KNN models for Swiggy delivery time prediction, evaluated using RMSE and R², achieving approximately 80–85% prediction accuracy.",
    "Kalyan used LangChain and Groq LLM to build the AI Gym Trainer Assistant, with an architecture supporting RAG, memory integration, and external tools.",
    "Kalyan holds a B.Tech in CSE (IoT) from Balaji Institute of Technology and Science (2022–2025) and is currently in the Data Science Program at Innomatics Research Labs.",
    "Kalyan built an ANN regression model with TensorFlow/Keras to predict mobile phone prices from hardware specifications, evaluated with MAE, MSE, RMSE, and R² Score.",
    "Kalyan's Gen AI and NLP skills include LLMs, Fine-Tuning, LangChain, NLTK, TF-IDF, RNN, LSTM, BERT, RAG, and VectorStores.",
    "Vamshi's XGBoost model for Swiggy delivery time prediction achieved R² = 0.7246 and RMSE = 4.9177, the best among the compared models.",
    "Vamshi built a content-based movie recommendation system using TMDB metadata, CountVectorizer (Bag-of-Words), and cosine similarity to generate top-N recommendations across ~5K+ movies.",
    "Vamshi's virtual keyboard using OpenCV and MediaPipe achieved 90%+ key detection accuracy at 25-30 FPS, reducing accidental key presses by 40%.",
    "Vamshi is undertaking Data Analytics and Data Science Training at Innomatics Research Labs, pursuing a Master of Computer Applications at NRIM Institute of Management and Science, and holds a BSc (MSTDS) from Satavahana University (09/2022–05/2025).",
    "During her internship at Auzmor Technology, Lekshana built a content-based Course Recommendation Engine using TF-IDF vectorization and cosine similarity across 3,000 Udemy courses, exposed via Dify, reducing manual course search effort by ~70%.",
    "Lekshana holds a Master of Artificial Intelligence and Data Science from Karunya Institute of Technology and Sciences (2023–2025, CGPA 8.0) and a Bachelor of Computer Science from Marudhar Kesari Jain College (2020–2023, CGPA 8.6).",
    "In RupeeRadar, Lekshana designed a multi-agent AI system using Model Context Protocol (MCP) to orchestrate market, sentiment, tax, and arbitrage agents with parallel execution and dynamic tool selection, integrating LLMs with CoinGecko, NewsAPI, and exchange APIs.",
    "Kushal's Random Forest and XGBoost models for delivery time prediction achieved an R² score of ~0.72 on test data, using 41K+ records and 20+ features.",
    "Kushal built a real-time face monitoring system using Python, OpenCV, and MediaPipe for online proctoring, detecting no-face and multiple-face scenarios with rule-based alerts and dynamic status updates.",
    "Kushal holds a Bachelor of Technology in Artificial Intelligence and Data Science from Vignan Institute of Technology and Science, Hyderabad (2021–2025)."
]

In [38]:
context = []
generated_answer = []

for query in questions:
    retrieved_docs = retriever_similarity.invoke(query)

    ## preparing context
    context_text = "\n\n".join(
        f"[Source: {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in retrieved_docs
    )
    context.append([doc.page_content for doc in retrieved_docs])

    ## preparing the answer
    prompt = f""" You are a professional AI resume retrieval assistant.you are provided with resume excerpts from vectore store.
    thes are excepts are only source of infromation.Generate as per context.

    Retrieved Resume Context:{context_text}

    Question:{query}

    Grounded Anmswer
    """
    result = model.invoke(prompt)
    generated_answer.append(result["answer"] if isinstance(result, dict) else result.content)

In [39]:
from datasets import Dataset

In [45]:
df = Dataset.from_dict({
    "question": questions,
    "contexts": context,
    "response": generated_answer,
    "ground_truth": ground_truths
})

In [46]:
from ragas.llms import llm_factory

In [48]:
llm2 = ChatOpenAI(model='gpt-4o-mini',api_key=OPENAI_API_KEY)

In [50]:
result = evaluate(llm = llm2,dataset=df,embeddings=embedding)

Evaluating: 100%|██████████| 200/200 [03:26<00:00,  1.03s/it]


In [51]:
data=result.to_pandas()

In [52]:
data.mean(numeric_only=True)

answer_relevancy     0.597544
context_precision    0.484283
faithfulness         0.710124
context_recall       0.573333
dtype: float64